# Kaggle Data Loader Configuration

In [ ]:
!pip install kaggle

In [ ]:
import os
import shutil

kaggle_api_path = "/content/drive/MyDrive/NIPS25/notebooks/kaggle.json"  # Update with your actual path

# Ensure ~/.kaggle directory exists
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

# Move kaggle.json to ~/.kaggle
shutil.move(kaggle_api_path, os.path.expanduser("~/.kaggle/kaggle.json"))

# Set proper permissions
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

print("✅ Kaggle API Key has been configured correctly!")


✅ Kaggle API Key has been configured correctly!


In [ ]:
!kaggle datasets download -d hussameldinanwer/mimic-iii --unzip

Dataset URL: https://www.kaggle.com/datasets/hussameldinanwer/mimic-iii
License(s): unknown
 93% 93.0M/100M [00:00<00:00, 171MB/s]
100% 100M/100M [00:00<00:00, 166MB/s] 


In [ ]:
import pandas as pd
df = pd.read_csv("/content/PRESCRIPTIONS.csv")  # Update filename if necessary

df.head()

<ipython-input-6-648b39076f22>:2: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/PRESCRIPTIONS.csv")  # Update filename if necessary


,ROW_ID,SUBJECT_ID,HADM_ID,ICUSTAY_ID,STARTDATE,ENDDATE,DRUG_TYPE,DRUG,DRUG_NAME_POE,DRUG_NAME_GENERIC,FORMULARY_DRUG_CD,GSN,NDC,PROD_STRENGTH,DOSE_VAL_RX,DOSE_UNIT_RX,FORM_VAL_DISP,FORM_UNIT_DISP,ROUTE
0,2214776,6,107064,NaN,2175-06-11 00:00:00,2175-06-12 00:00:00,MAIN,Tacrolimus,Tacrolimus,Tacrolimus,TACR1,021796,469061711.0,1mg Capsule,2,mg,2,CAP,PO
1,2214775,6,107064,NaN,2175-06-11 00:00:00,2175-06-12 00:00:00,MAIN,Warfarin,Warfarin,Warfarin,WARF5,006562,56017275.0,5mg Tablet,5,mg,1,TAB,PO
2,2215524,6,107064,NaN,2175-06-11 00:00:00,2175-06-12 00:00:00,MAIN,Heparin Sodium,NaN,NaN,HEPAPREMIX,006522,338055002.0,"25,000 unit Premix Bag","25,000",UNIT,1,BAG,IV
3,2216265,6,107064,NaN,2175-06-11 00:00:00,2175-06-12 00:00:00,BASE,D5W,NaN,NaN,HEPBASE,NaN,0.0,HEPARIN BASE,250,ml,250,ml,IV
4,2214773,6,107064,NaN,2175-06-11 00:00:00,2175-06-12 00:00:00,MAIN,Furosemide,Furosemide,Furosemide,FURO20,008208,54829725.0,20mg Tablet,20,mg,1,TAB,PO


# API Configuration

In [ ]:
import json
import requests
import torch
import numpy as np
from scipy.special import softmax
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch import nn
from openai import OpenAI

import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd

In [ ]:
client = OpenAI(api_key="REDACTED_USE_ENV_OPENAI_API_KEY")  # Replace with actual API key

# GPT Models on MIMIC-III

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
import pandas as pd
import requests
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.special import softmax
from openai import OpenAI


# ✅ Load MIMIC-III Dataset (Modify Path as Needed)
df = pd.read_csv("/content/PRESCRIPTIONS.csv")

# ✅ Extract Clinical Notes or Relevant Text Fields
questions = df["DRUG_NAME_GENERIC"].dropna().astype(str).tolist()
num_samples = min(10, len(questions))  # Limit to 10 samples for evaluation

# ✅ Define Models
MODELS = {
    "GPT-4.5": "gpt-4.5-preview",
    "GPT-4o": "gpt-4o",
    "GPT-4": "gpt-4",
    "GPT-3.5": "gpt-3.5-turbo"
}

# ✅ Load GPT-2 for Local Testing
tokenizer_gpt2 = AutoTokenizer.from_pretrained("gpt2")
tokenizer_gpt2.pad_token = tokenizer_gpt2.eos_token  # Fix padding
model_gpt2 = AutoModelForCausalLM.from_pretrained("gpt2")

# ✅ BayesianGPT Class (MC Dropout)
class BayesianGPT(nn.Module):
    def __init__(self):
        super(BayesianGPT, self).__init__()
        self.dropout = nn.Dropout(p=0.3)  # Apply dropout to simulate Bayesian uncertainty

    def forward(self, input_text, model_name, n_samples=10):
        """Simulate Bayesian inference with multiple API calls (Monte Carlo Estimation)."""
        predictions = []

        for _ in range(n_samples):
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": input_text}],
                max_tokens=50,
                temperature=np.random.uniform(0.7, 1.2)  # Randomized temp for diversity
            )
            predictions.append(len(response.choices[0].message.content.split()))

        mean_prediction = np.mean(predictions)
        uncertainty = np.abs(np.std(predictions) / (mean_prediction + 1e-6))  # Normalize
        return mean_prediction, uncertainty

# ✅ Uncertainty-Weighted Attention Mechanism
def apply_uncertainty_weighted_attention(uncertainties, logits):
    weights = np.exp(-uncertainties)  # Higher uncertainty -> Lower weight
    weighted_logits = logits * weights
    return weighted_logits

# ✅ Function: Generate Response (No Uncertainty) - OpenAI GPT
def generate_response_no_uncertainty(input_text, model_name):
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": input_text}],
        max_tokens=50,
        temperature=1.0
    )
    return response.choices[0].message.content

# ✅ Function: Generate Response (No Uncertainty) - GPT-2
def generate_response_no_uncertainty_gpt2(input_text):
    inputs = tokenizer_gpt2(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    output_sequences = model_gpt2.generate(inputs["input_ids"], max_new_tokens=50, pad_token_id=tokenizer_gpt2.eos_token_id)
    response = tokenizer_gpt2.decode(output_sequences[0], skip_special_tokens=True)
    return response

# ✅ Function: Bayesian Uncertainty Estimation (GPT-4.5, GPT-4o, GPT-4, GPT-3.5)
bayesian_model = BayesianGPT()

# ✅ Function: Estimate Uncertainty for GPT-2
def estimate_uncertainty_gpt2(input_text, n_samples=10):
    inputs = tokenizer_gpt2(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    input_ids = inputs["input_ids"]
    predictions = []

    for _ in range(n_samples):
        with torch.no_grad():
            output_sequences = model_gpt2.generate(input_ids, max_new_tokens=50, pad_token_id=tokenizer_gpt2.eos_token_id)
            response = tokenizer_gpt2.decode(output_sequences[0], skip_special_tokens=True)
            predictions.append(len(response.split()))

    mean_prediction = np.mean(predictions)
    uncertainty = np.abs(np.std(predictions) / (mean_prediction + 1e-6))  # Normalize
    return mean_prediction, uncertainty

# ✅ Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    return np.mean(uncertainties)

# ✅ Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    if len(cus_scores) < 2:
        return 1.0  # Default if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# ✅ Confidence-Guided Decision Shaping
def refine_decision_with_confidence(logits, uncertainty):
    threshold = 0.5  # Confidence threshold
    refined_logits = np.where(uncertainty < threshold, logits, logits * 0.8)  # Reduce low-confidence logits
    return refined_logits

# ✅ Run Evaluation for GPT-4.5, GPT-4o, GPT-4, GPT-3.5, and GPT-2
results = {}

for model_name, model_key in MODELS.items():
    print(f"\n====== Running {model_name} on MIMIC-III ======")

    cus_scores_no_uncertainty = []
    cus_scores_with_uncertainty = []

    for i in range(num_samples):
        # 🔹 Without Uncertainty
        response_no_uncertainty = generate_response_no_uncertainty(questions[i], model_key)
        print(f"Sample {i+1} - {model_name} Response (No Uncertainty): {response_no_uncertainty}")

        cus_score_no_uncertainty = len(response_no_uncertainty.split())  # Length-based proxy
        cus_scores_no_uncertainty.append(cus_score_no_uncertainty)

        # 🔹 With Bayesian Uncertainty Modeling
        mean_pred, uncertainty = bayesian_model(questions[i], model_key, n_samples=10)
        cus_score = compute_cus([uncertainty])
        cus_scores_with_uncertainty.append(cus_score)

        print(f"Sample {i+1} - {model_name} CUS (With Uncertainty): {cus_score}")

    # ✅ Compute Overall ZTI
    zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
    zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)

    print(f"{model_name} Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
    print(f"{model_name} Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

    # ✅ Store Results
    results[model_name] = {
        "CUS_No_Uncertainty": cus_scores_no_uncertainty,
        "CUS_With_Uncertainty": cus_scores_with_uncertainty,
        "ZTI_No_Uncertainty": zti_score_no_uncertainty,
        "ZTI_With_Uncertainty": zti_score_with_uncertainty,
    }

# ✅ Run Evaluation for GPT-2
print(f"\n====== Running GPT-2 on MIMIC-III ======")
cus_scores_no_uncertainty = []
cus_scores_with_uncertainty = []

for i in range(num_samples):
    response_no_uncertainty = generate_response_no_uncertainty_gpt2(questions[i])
    mean_pred, uncertainty = estimate_uncertainty_gpt2(questions[i])

    cus_scores_no_uncertainty.append(len(response_no_uncertainty.split()))
    cus_scores_with_uncertainty.append(compute_cus([uncertainty]))

results["GPT-2"] = {
    "CUS_No_Uncertainty": cus_scores_no_uncertainty,
    "CUS_With_Uncertainty": cus_scores_with_uncertainty,
    "ZTI_No_Uncertainty": compute_zti(cus_scores_no_uncertainty),
    "ZTI_With_Uncertainty": compute_zti(cus_scores_with_uncertainty),
}

# ✅ Print Final Results
for model_name, metrics in results.items():
    print(f"\nModel: {model_name}")
    print("CUS (No Uncertainty):", metrics["CUS_No_Uncertainty"])
    print("CUS (With Uncertainty):", metrics["CUS_With_Uncertainty"])
    print("ZTI (No Uncertainty):", metrics["ZTI_No_Uncertainty"])
    print("ZTI (With Uncertainty):", metrics["ZTI_With_Uncertainty"])


<ipython-input-9-ea95cb91b740>:13: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/PRESCRIPTIONS.csv")
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


====== Running GPT-4.5 on MIMIC-III ======
Sample 1 - GPT-4.5 Response (No Uncertainty): Tacrolimus is an immunosuppressive medication commonly used to prevent organ rejection after transplantation and to treat severe autoimmune disorders. It belongs to a class of drugs called calcineurin inhibitors.

Mechanism of Action:
Tacrolimus inhibits calcineurin
Sample 1 - GPT-4.5 CUS (With Uncertainty): 0.09240579901327993
Sample 2 - GPT-4.5 Response (No Uncertainty): Warfarin is an anticoagulant medication commonly used to prevent and treat blood clots. It helps reduce clot formation, lowering the risk of complications like stroke, pulmonary embolism (clots in the lungs), deep vein thrombosis (DVT
Sample 2 - GPT-4.5 CUS (With Uncertainty): 0.06275036672844593
Sample 3 - GPT-4.5 Response (No Uncertainty): Furosemide, commonly marketed under brand names such as Lasix, is a powerful loop diuretic medication used primarily to reduce fluid buildup (edema) caused by various conditions. It works by

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Sample 10 - GPT-3.5 CUS (With Uncertainty): 0.043315385480843294
GPT-3.5 Zero-Shot Trustworthiness Index (ZTI) No Uncertainty: 0.036587834418470355
GPT-3.5 Zero-Shot Trustworthiness Index (ZTI) With Uncertainty: 0.4058431034137958

====== Running GPT-2 on MIMIC-III ======

Model: GPT-4.5
CUS (No Uncertainty): [35, 35, 35, 34, 39, 38, 35, 32, 36, 37]
CUS (With Uncertainty): [0.09240579901327993, 0.06275036672844593, 0.04325615351021454, 0.03706191146347384, 0.03400154672902016, 0.08977298220085289, 0.06911725173022847, 0.06048122635016408, 0.07358206802116268, 0.07743306270937195]
ZTI (No Uncertainty): 0.038593518172549524
ZTI (With Uncertainty): 0.5085589430252958

Model: GPT-4o
CUS (No Uncertainty): [37, 40, 37, 38, 35, 37, 34, 37, 34, 35]
CUS (With Uncertainty): [0.04414451452382779, 0.045685846824507687, 0.05992776094368123, 0.046405718608321135, 0.05095394637761523, 0.02212535256275674, 0.06000175982127443, 0.03351121682350124, 0.04887684671964996, 0.03726779858977987]
ZTI (No Unce

# BERT Models on MIMIC III

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax

# ✅ Load MIMIC-III Dataset (Modify Path as Needed)
df = pd.read_csv("/content/PRESCRIPTIONS.csv", low_memory=False)  # Fix mixed dtype warning

# ✅ Extract Clinical Notes or Relevant Text Fields
questions = df["DRUG_NAME_GENERIC"].dropna().astype(str).tolist()
num_samples = min(10, len(questions))  # Limit to 10 samples for evaluation

# ✅ Define BERT-based Models
BERT_MODELS = {
    "BERT-Base": "bert-base-uncased",
    "BERT-Large": "bert-large-uncased",
    "BioBERT": "dmis-lab/biobert-base-cased-v1.1",
    "ClinicalBERT": "emilyalsentzer/Bio_ClinicalBERT",
    "RoBERTa": "roberta-base"
}

# ✅ BayesianBERT Class (MC Dropout)
class BayesianBERT(nn.Module):
    def __init__(self, base_model):
        super(BayesianBERT, self).__init__()
        self.base_model = base_model
        self.dropout = nn.Dropout(p=0.3)  # Apply Bayesian uncertainty

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.base_model(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        logits = self.dropout(outputs.logits)  # Dropout during inference
        return logits

# ✅ Function: Predict with BERT Models (No Uncertainty)
def predict_no_uncertainty_bert(model, tokenizer, input_text):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    if "token_type_ids" not in inputs:
        inputs["token_type_ids"] = None  # Handle models like RoBERTa that do not use token_type_ids

    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    probs = softmax(logits.numpy(), axis=1)
    return probs

# ✅ Function: Bayesian Uncertainty Estimation (BERT)
def estimate_uncertainty_bert(model, tokenizer, input_text, n_samples=10):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    if "token_type_ids" not in inputs:
        inputs["token_type_ids"] = None

    predictions = []
    for _ in range(n_samples):
        with torch.no_grad():
            logits = model(**inputs)
            predictions.append(logits.numpy())

    predictions = np.array(predictions)
    mean_prediction = predictions.mean(axis=0)
    uncertainty = np.abs(predictions.std(axis=0) / (mean_prediction + 1e-6))  # Normalize uncertainty
    return mean_prediction, uncertainty

# ✅ Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    return np.mean(uncertainties)

# ✅ Improved Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    if len(cus_scores) < 2:
        return 1.0  # Default if only one sample is available

    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)  # Ensure stability

    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))
    return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# ✅ Evaluate BERT Models
results = {}

for model_name, model_path in BERT_MODELS.items():
    print(f"\n====== Running {model_name} on MIMIC-III ======")

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, num_labels=4, ignore_mismatched_sizes=True
    )
    bayesian_model = BayesianBERT(model)

    cus_scores_no_uncertainty = []
    cus_scores_with_uncertainty = []

    for i in range(num_samples):
        probs_no_uncertainty = predict_no_uncertainty_bert(model, tokenizer, questions[i])
        cus_scores_no_uncertainty.append(compute_cus(probs_no_uncertainty))

        pred, uncertainty = estimate_uncertainty_bert(bayesian_model, tokenizer, questions[i])
        cus_scores_with_uncertainty.append(compute_cus(uncertainty))

    # ✅ Compute Overall ZTI
    zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
    zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)

    print(f"{model_name} Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
    print(f"{model_name} Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

    # ✅ Store Results
    results[model_name] = {
        "CUS_No_Uncertainty": cus_scores_no_uncertainty,
        "CUS_With_Uncertainty": cus_scores_with_uncertainty,
        "ZTI_No_Uncertainty": zti_score_no_uncertainty,
        "ZTI_With_Uncertainty": zti_score_with_uncertainty,
    }

# ✅ Final Comparison Report
print("\n===== Final Comparison Report =====")
for model_name, metrics in results.items():
    print(f"\nModel: {model_name}")
    print("CUS (No Uncertainty):", metrics["CUS_No_Uncertainty"])
    print("CUS (With Uncertainty):", metrics["CUS_With_Uncertainty"])
    print("ZTI (No Uncertainty):", metrics["ZTI_No_Uncertainty"])
    print("ZTI (With Uncertainty):", metrics["ZTI_With_Uncertainty"])



====== Running BERT-Base on MIMIC-III ======


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERT-Base Zero-Shot Trustworthiness Index (ZTI) No Uncertainty: 0
BERT-Base Zero-Shot Trustworthiness Index (ZTI) With Uncertainty: 0.3035917282104492

====== Running BERT-Large on MIMIC-III ======


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

... (20 outputs left out)

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RoBERTa Zero-Shot Trustworthiness Index (ZTI) No Uncertainty: 0
RoBERTa Zero-Shot Trustworthiness Index (ZTI) With Uncertainty: 0.24863272905349731

===== Final Comparison Report =====

Model: BERT-Base
CUS (No Uncertainty): [0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25]
CUS (With Uncertainty): [0.8341917, 0.78145075, 0.53566754, 0.57611954, 0.8329964, 0.61779034, 0.74278814, 0.41366884, 0.5773261, 0.6624527]
ZTI (No Uncertainty): 0
ZTI (With Uncertainty): 0.3035917282104492

Model: BERT-Large
CUS (No Uncertainty): [0.24999999, 0.25, 0.25000003, 0.25, 0.25, 0.24999999, 0.24999999, 0.25, 0.24999999, 0.24999999]
CUS (With Uncertainty): [0.6969107, 0.9311849, 0.53145003, 0.86237127, 0.3719964, 0.79897326, 0.6546529, 0.88351226, 0.7832469, 0.7152128]
ZTI (No Uncertainty): 1.1920928955078125e-07
ZTI (With Uncertainty): 0.32363665103912354

Model: BioBERT
CUS (No Uncertainty): [0.25, 0.25, 0.25000003, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25, 0.25]
CUS (With Uncertainty): [0.65465593

# LLaMa Models on MIMIC-III

In [ ]:
! huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `mimicdatatoken` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `mim

In [ ]:
! pip install transformers accelerate torch bitsandbytes peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 785.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [ ]:
! pip install -U bitsandbytes

In [ ]:
! pip install "transformers>=4.45.1"

In [ ]:
# import torch
# import torch.nn as nn
# import numpy as np
# import pandas as pd
# from transformers import AutoTokenizer, AutoModelForCausalLM
# from peft import get_peft_model, LoraConfig, TaskType
# from scipy.special import softmax

# # ✅ Load MIMIC-III Dataset (Modify Path as Needed)
# df = pd.read_csv("/content/PRESCRIPTIONS.csv")

# # ✅ Extract Clinical Notes or Relevant Text Fields
# questions = df["DRUG_NAME_GENERIC"].dropna().astype(str).tolist()
# num_samples = min(10, len(questions))  # Limit to 10 samples for evaluation

# # ✅ Define LLaMA-2 Models
# LLAMA_MODELS = {
#     "LLaMA-2-7B": "meta-llama/Llama-2-7b-hf",
#     "LLaMA-2-13B": "meta-llama/Llama-2-13b-hf"
# }

# # ✅ LoRA Configuration
# lora_config = LoraConfig(
#     r=8,  # Low-rank adaptation factor
#     lora_alpha=32,  # Scaling factor
#     lora_dropout=0.1,  # Dropout for LoRA
#     task_type=TaskType.CAUSAL_LM,  # Adapting for causal language modeling
# )

# # ✅ BayesianLLaMA Class (MC Dropout + LoRA)
# class BayesianLLaMA(nn.Module):
#     def __init__(self, base_model):
#         super(BayesianLLaMA, self).__init__()
#         self.base_model = get_peft_model(base_model, lora_config)  # Apply LoRA
#         self.dropout = nn.Dropout(p=0.3)  # Bayesian Uncertainty

#     def forward(self, input_ids, attention_mask):
#         outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
#         logits = self.dropout(outputs.logits)  # Apply MC Dropout
#         return logits

# # ✅ Function: Generate Response (LoRA)
# def generate_response_lora(model, tokenizer, input_text):
#     inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
#     with torch.no_grad():
#         output_sequences = model.generate(inputs["input_ids"], max_new_tokens=50, pad_token_id=tokenizer.pad_token_id)
#     response = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
#     return response

# # ✅ Function: Bayesian Uncertainty Estimation (LoRA)
# def estimate_uncertainty_lora(model, tokenizer, input_text, n_samples=10):
#     inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
#     predictions = []

#     for _ in range(n_samples):
#         with torch.no_grad():
#             logits = model(inputs["input_ids"], inputs["attention_mask"])
#             predictions.append(logits.cpu().numpy())

#     predictions = np.array(predictions)
#     mean_prediction = predictions.mean(axis=0)
#     uncertainty = np.abs(predictions.std(axis=0) / (mean_prediction + 1e-6))  # Normalize uncertainty
#     return mean_prediction, uncertainty

# # ✅ Uncertainty-Weighted Attention Mechanism
# def apply_uncertainty_weighted_attention(uncertainties, logits):
#     weights = np.exp(-uncertainties)  # Higher uncertainty → Lower weight
#     weighted_logits = logits * weights
#     return weighted_logits

# # ✅ Confidence-Guided Decision Shaping
# def refine_decision_with_confidence(logits, uncertainty):
#     threshold = 0.5  # Confidence threshold
#     refined_logits = np.where(uncertainty < threshold, logits, logits * 0.8)  # Reduce low-confidence logits
#     return refined_logits

# # ✅ Clinical Uncertainty Score (CUS)
# def compute_cus(uncertainties):
#     return np.mean(uncertainties)

# # ✅ Improved Zero-Shot Trustworthiness Index (ZTI)
# def compute_zti(cus_scores, epsilon=1e-6):
#     if len(cus_scores) < 2:
#         return 1.0  # Default if only one sample is available

#     mean_cus = np.mean(cus_scores)
#     std_cus = np.std(cus_scores)
#     max_cus = max(np.max(cus_scores), epsilon)  # Ensure stability

#     zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))
#     return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# # ✅ Run Evaluation for LLaMA-2 Models with LoRA
# results = {}

# for model_name, model_path in LLAMA_MODELS.items():
#     print(f"\n====== Running {model_name} with LoRA on MIMIC-III ======")

#     tokenizer = AutoTokenizer.from_pretrained(model_path)
#     tokenizer.pad_token = tokenizer.eos_token  # Ensure correct padding
#     base_model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")

#     # ✅ Apply LoRA & Bayesian Uncertainty
#     bayesian_model = BayesianLLaMA(base_model)
#     bayesian_model.train()  # Ensure LoRA dropout is active

#     cus_scores_no_uncertainty = []
#     cus_scores_with_uncertainty = []

#     for i in range(num_samples):
#         # 🔹 Without Uncertainty
#         response_no_uncertainty = generate_response_lora(base_model, tokenizer, questions[i])
#         print(f"Sample {i+1} - {model_name} Response (No Uncertainty): {response_no_uncertainty}")

#         cus_score_no_uncertainty = len(response_no_uncertainty.split())  # Length-based proxy
#         cus_scores_no_uncertainty.append(cus_score_no_uncertainty)

#         # 🔹 With Bayesian Uncertainty Modeling
#         mean_pred, uncertainty = estimate_uncertainty_lora(bayesian_model, tokenizer, questions[i], n_samples=10)
#         cus_score = compute_cus([uncertainty])
#         cus_scores_with_uncertainty.append(cus_score)

#         print(f"Sample {i+1} - {model_name} CUS (With Uncertainty - LoRA): {cus_score}")

#     # ✅ Compute Overall ZTI
#     zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
#     zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)

#     print(f"{model_name} Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
#     print(f"{model_name} Zero-Shot Trustworthiness Index (ZTI) With Uncertainty (LoRA):", zti_score_with_uncertainty)

#     # ✅ Store Results
#     results[model_name] = {
#         "CUS_No_Uncertainty": cus_scores_no_uncertainty,
#         "CUS_With_Uncertainty": cus_scores_with_uncertainty,
#         "ZTI_No_Uncertainty": zti_score_no_uncertainty,
#         "ZTI_With_Uncertainty": zti_score_with_uncertainty,
#     }

# # ✅ Final Comparison Report
# print("\n===== Final Comparison Report =====")
# for model_name, metrics in results.items():
#     print(f"\nModel: {model_name}")
#     print("CUS (No Uncertainty):", metrics["CUS_No_Uncertainty"])
#     print("CUS (With Uncertainty - LoRA):", metrics["CUS_With_Uncertainty"])
#     print("ZTI (No Uncertainty):", metrics["ZTI_No_Uncertainty"])
#     print("ZTI (With Uncertainty - LoRA):", metrics["ZTI_With_Uncertainty"])


In [ ]:
# import torch
# import torch.nn as nn
# import numpy as np
# import json
# import pandas as pd
# from transformers import AutoTokenizer, AutoModelForCausalLM

# # ✅ Load MIMIC-III Dataset (Modify Path as Needed)
# df = pd.read_csv("/content/PRESCRIPTIONS.csv")

# # ✅ Extract Clinical Notes or Relevant Text Fields
# questions = df["DRUG_NAME_GENERIC"].dropna().astype(str).tolist()
# num_samples = min(10, len(questions))  # Limit to 10 samples for evaluation

# # ✅ Define LLaMA-2 Models
# LLAMA_MODELS = {
#     "LLaMA-2-7B": "meta-llama/Llama-2-7b-hf",
#     "LLaMA-2-13B": "meta-llama/Llama-2-13b-hf"
# }

# # ✅ Bayesian LLaMA Class (MC Dropout)
# class BayesianLLaMA(nn.Module):
#     def __init__(self, base_model):
#         super(BayesianLLaMA, self).__init__()
#         self.base_model = base_model
#         self.dropout = nn.Dropout(p=0.3)  # Apply Bayesian dropout

#     def forward(self, input_ids, attention_mask):
#         outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
#         logits = self.dropout(outputs.logits)  # Dropout for Bayesian uncertainty
#         return logits

# # ✅ Function: Generate Response (No Uncertainty)
# def generate_response_no_uncertainty(model, tokenizer, input_text):
#     inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
#     with torch.no_grad():
#         output_sequences = model.generate(inputs["input_ids"], max_new_tokens=50, pad_token_id=tokenizer.pad_token_id)
#     response = tokenizer.decode(output_sequences[0], skip_special_tokens=True)
#     return response

# # ✅ Function: Bayesian Uncertainty Estimation (MC Dropout)
# def estimate_uncertainty_llama(model, tokenizer, input_text, n_samples=10):
#     inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
#     predictions = []

#     for _ in range(n_samples):
#         with torch.no_grad():
#             logits = model(inputs["input_ids"], inputs["attention_mask"])
#             predictions.append(logits.cpu().numpy())

#     predictions = np.array(predictions)
#     mean_prediction = predictions.mean(axis=0)
#     uncertainty = np.abs(predictions.std(axis=0) / (mean_prediction + 1e-6))  # Normalize uncertainty
#     return mean_prediction, uncertainty

# # ✅ Clinical Uncertainty Score (CUS)
# def compute_cus(uncertainties):
#     return np.mean(uncertainties)

# # ✅ Improved Zero-Shot Trustworthiness Index (ZTI)
# def compute_zti(cus_scores, epsilon=1e-6):
#     if len(cus_scores) < 2:
#         return 1.0  # Default if only one sample is available

#     mean_cus = np.mean(cus_scores)
#     std_cus = np.std(cus_scores)
#     max_cus = max(np.max(cus_scores), epsilon)  # Ensure stability

#     zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))
#     return max(0, min(zti, 1))  # Ensure ZTI stays within [0,1] range

# # ✅ Run Evaluation for LLaMA-2 (7B & 13B) on CPU
# results = {}

# for model_name, model_path in LLAMA_MODELS.items():
#     print(f"\n====== Running {model_name} on MIMIC-III (CPU) ======")

#     # ✅ Load Tokenizer & Model (Force CPU Execution)
#     tokenizer = AutoTokenizer.from_pretrained(model_path)
#     model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float32, device_map=None).to("cpu")
#     bayesian_model = BayesianLLaMA(model)

#     cus_scores_no_uncertainty = []
#     cus_scores_with_uncertainty = []

#     for i in range(num_samples):
#         # 🔹 Without Uncertainty
#         response_no_uncertainty = generate_response_no_uncertainty(model, tokenizer, questions[i])
#         print(f"Sample {i+1} - {model_name} Response (No Uncertainty): {response_no_uncertainty}")

#         cus_score_no_uncertainty = len(response_no_uncertainty.split())  # Length-based proxy
#         cus_scores_no_uncertainty.append(cus_score_no_uncertainty)

#         # 🔹 With Bayesian Uncertainty Modeling
#         mean_pred, uncertainty = estimate_uncertainty_llama(bayesian_model, tokenizer, questions[i])
#         cus_score = compute_cus([uncertainty])
#         cus_scores_with_uncertainty.append(cus_score)

#         print(f"Sample {i+1} - {model_name} CUS (With Uncertainty): {cus_score}")

#     # ✅ Compute Overall ZTI
#     zti_score_no_uncertainty = compute_zti(cus_scores_no_uncertainty)
#     zti_score_with_uncertainty = compute_zti(cus_scores_with_uncertainty)

#     print(f"{model_name} Zero-Shot Trustworthiness Index (ZTI) No Uncertainty:", zti_score_no_uncertainty)
#     print(f"{model_name} Zero-Shot Trustworthiness Index (ZTI) With Uncertainty:", zti_score_with_uncertainty)

#     # ✅ Store Results
#     results[model_name] = {
#         "CUS_No_Uncertainty": cus_scores_no_uncertainty,
#         "CUS_With_Uncertainty": cus_scores_with_uncertainty,
#         "ZTI_No_Uncertainty": zti_score_no_uncertainty,
#         "ZTI_With_Uncertainty": zti_score_with_uncertainty,
#     }

# # ✅ Final Comparison Report
# print("\n===== Final Comparison Report =====")
# for model_name, metrics in results.items():
#     print(f"\nModel: {model_name}")
#     print("CUS (No Uncertainty):", metrics["CUS_No_Uncertainty"])
#     print("CUS (With Uncertainty):", metrics["CUS_With_Uncertainty"])
#     print("ZTI (No Uncertainty):", metrics["ZTI_No_Uncertainty"])
#     print("ZTI (With Uncertainty):", metrics["ZTI_With_Uncertainty"])
